# How Much of a Transformer Is Code? — 4B substitution

Measures the **thesis-critical number the paper is missing at scale**: how much loss it costs to replace ~36% of attention heads (positional templates) + ~21% of MLP layers (token lookup tables) in **Qwen3-4B**, after healing only the RMSNorm gains — versus the ~+0.88 nat fair cost measured at 0.6B.

**Before running:** `Runtime → Change runtime type → GPU` (an **L4 or A100 / 24 GB+** — the free T4 will likely OOM on the 4B heal).

Then `Runtime → Run all`. It auto-pulls WikiText-103 (no upload) and prints the headline block.

Repo: https://github.com/keithagroves/how-much-transformer-is-code

In [ ]:
!pip install -q transformers datasets accelerate
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU — set Runtime to GPU')

In [ ]:
#@title Settings { run: "auto" }
MODEL = "Qwen/Qwen3-4B"  #@param ["Qwen/Qwen3-4B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-0.6B"]
HEAD_FRACTION = 0.36  #@param {type:"slider", min:0.1, max:0.5, step:0.02}
HEAL_EPOCHS = 12  #@param {type:"slider", min:4, max:20, step:1}
print(MODEL, HEAD_FRACTION, HEAL_EPOCHS)

In [ ]:
# pull the latest pipeline script from the repo and run it
!wget -q -O sub.py https://raw.githubusercontent.com/keithagroves/how-much-transformer-is-code/main/src/colab_4b_substitution.py
!python sub.py "$MODEL" "$HEAD_FRACTION" "$HEAL_EPOCHS"

### Reading the output
- **code + heal** and **fair cost** (net of the intact-heal domain-adaptation offset) are the numbers to compare to 0.6B's **~+0.88 nats**.
- **zero-ablation + heal** is the control — code should sit well below it, or the substitution isn't carrying function.
- If code+heal and fair cost land in the **+0.6–0.9** band, the headline is **not** a small-model artifact. Much lower/higher means scale shifts the substitutable fraction.

Note: this uses **positional-only templates** (content-free), justified because the paper's positional-vs-fitted sweep shows content columns buy ~nothing in the substitutable set.